# 02 周期材料结构：从晶胞到 pymatgen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/02_pymatgen_structure.ipynb)

这一章先不谈机器学习。目标是理解：一个 CIF 文件为什么能描述周期材料，以及 Python 如何读取晶胞和原子坐标。

## COF 背景衔接：孔道示意图不等于晶胞

二维 COF 的构筑单元在层内形成共价网络，实际三维晶体还包含层与层的排列；三维 COF 则通过共价键向三个空间方向延伸。一个孔的轮廓并不能完整指定晶胞，还需要晶格矢量、原子坐标和周期镜像才能重建结构。

回看 [COF 背景图](https://raw.githubusercontent.com/Wanteen/COF-ML-Tutorial/main/assets/cof_background.jpg)，尝试区分分子构筑单元、孔道和层间排列。下文手动构造的四原子结构仅用于学习 API，不是化学上完整的 COF，也不对应图中的具体材料；分析真实材料时应使用经过检查的 CIF。

## 1. 晶胞由什么决定？
一个一般晶胞由 6 个晶格参数描述：
- `a, b, c`：三条晶格边长度；
- `α, β, γ`：三组夹角。

**注意：COF 并不要求 `a=b`。** 有些六方/三方对称结构满足 `a=b`，但矩形、斜方、单斜或经过优化后的 COF 都可能出现 `a ≠ b`。因此先学习一般晶胞，再把 hexagonal 当作特殊情况。

In [ ]:
!pip -q install pymatgen

In [ ]:
from pymatgen.core import Lattice, Structure

# 一般晶胞：a、b、c 可以不同，角度也可以不是 90°
lattice = Lattice.from_parameters(
    a=18.0, b=22.0, c=3.6,
    alpha=90, beta=90, gamma=95
)
print(lattice)
print('a, b, c =', lattice.a, lattice.b, lattice.c)
print('alpha, beta, gamma =', lattice.alpha, lattice.beta, lattice.gamma)
print('volume =', lattice.volume)

## 2. Hexagonal 是特殊情况
六方晶胞通常满足 `a=b`，且 `γ=120°`。pymatgen 提供快捷写法：

In [ ]:
hex_lattice = Lattice.hexagonal(a=12.0, c=3.5)
print(hex_lattice)
print('a =',hex_lattice.a,'b =',hex_lattice.b,'gamma =',hex_lattice.gamma)

## 3. 原子放在哪里？
周期结构常用 **fractional coordinates（分数坐标）**。例如 `[0.5, 0.5, 0.5]` 表示沿三条晶格矢量各走一半。

分数坐标不是 Å；它是相对于晶胞的比例。pymatgen 可以自动转换成 Cartesian coordinates（笛卡尔坐标，单位 Å）。

In [ ]:
structure = Structure(
    lattice,
    ['C','C','N','N'],
    [[0,0,0.5],[0.5,0.5,0.5],[0.25,0.25,0.5],[0.75,0.75,0.5]]
)
print('Formula:',structure.composition.reduced_formula)
print('Number of atoms:',len(structure))
print('Volume (A^3):',structure.volume)
print('Density:',structure.density)
for i,site in enumerate(structure):
    print(i,site.species_string,'frac=',site.frac_coords,'cart=',site.coords)

## 4. 什么叫周期性？
晶胞会在空间重复。一个靠近晶胞右边界的原子，可能与相邻周期镜像中的原子很近。

这就是 **periodic boundary conditions（PBC，周期性边界条件）**。分析邻居时不能只看当前盒子里画出来的原子。

In [ ]:
center=structure[0]
neighbors=structure.get_neighbors(center,r=8.0)
for n in neighbors[:10]:
    print(n.species_string,'distance=',round(n.nn_distance,3),'image=',n.image)

## 5. 真实科研中通常直接读取 CIF
```python
from pymatgen.core import Structure
s = Structure.from_file('your_cof.cif')
```

读取后建议先检查：
- `a, b, c, α, β, γ`；
- atom count 和元素组成；
- 是否包含 guest / solvent；
- H 是否缺失；
- 是否存在 disorder / occupancy；
- 2D COF 的层间距离和 stacking 是否合理。

## 本章术语表
- lattice：晶格；
- unit cell：晶胞；
- fractional coordinates：分数坐标；
- Cartesian coordinates：笛卡尔坐标；
- PBC：周期性边界条件；
- CIF：晶体结构常用文件格式。

## Exercises
1. 把 `a=18` 改成 20，观察 volume。
2. 把 `b=22` 改成 18，比较 `a=b` 和 `a≠b`。
3. 把 `gamma=95` 改成 90，观察 volume。
4. 解释为什么 `Lattice.hexagonal()` 不适合表示所有 COF。
5. 解释 fractional coordinate `[0.5,0.5,0.5]` 的意义。

### 本章最低要求
知道一般晶胞有 6 个参数，并理解 `a`、`b` 不一定相等；知道 CIF 最终会被解析成晶格 + 原子 + 坐标。